In [0]:
%run ./config

##1. Configuração

##2. Schema do JSON do ClinicalTrials vem via notebook config

##3. Ler e deduplicar a Bronze

In [0]:
studies = (clinical_parsed
    .select(F.upper(F.trim(p["identificationModule"]["nctId"])).alias("nct_id"),
            F.regexp_replace(F.trim(p["identificationModule"]["briefTitle"]),r"\s+"," ").alias("study_title"),
            F.upper(p["designModule"]["studyType"]).alias("study_type"),
            F.upper(p["statusModule"]["overallStatus"]).alias("overall_status"),
            F.array_join(F.array_sort(p["designModule"]["phases"]),"|").alias("phase"),
                                      p["statusModule"]["startDateStruct"]["date"].alias("start_date_original"),
                                      p["statusModule"]["startDateStruct"]["type"].alias("start_date_type"),
                                      p["statusModule"]["completionDateStruct"]["date"].alias("completion_date_original"),
                                      p["statusModule"]["completionDateStruct"]["type"].alias("completion_date_type"),
                                      p["designModule"]["enrollmentInfo"]["count"].cast("long").alias("enrollment_count"),
                                      F.upper(p["designModule"]["enrollmentInfo"]["type"]).alias("enrollment_type"),
                                      F.regexp_replace(F.trim(p["sponsorCollaboratorsModule"]["leadSponsor"]["name"]),r"\s+"," ").alias("sponsor_name"),
                                      F.upper(p["sponsorCollaboratorsModule"]["leadSponsor"]["class"]).alias("sponsor_class"),"ingestion_id","collected_at")
    .withColumn("start_date",parse_partial_date(F.col("start_date_original")))
    .withColumn("completion_date",parse_partial_date(F.col("completion_date_original")))
    .withColumn("duration_days",F.when(F.col("completion_date") >= F.col("start_date"),F.datediff("completion_date", "start_date"))))

In [0]:
studies.write.format("delta")\
             .mode("overwrite")\
             .option("overwriteSchema", "true")\
             .saveAsTable(f"{CATALOG}.{SCHEMA}.slv_studies")

In [0]:
%sql
SELECT * FROM mvp_eng_dados.mvp_cancer.slv_studies